# 01b — Flood mapping with SAR: histograms, thresholds and Otsu
**Event:** EMSR271_02FARKADONA (Thessaly, Greece). Sentinel-1 GRD (IW, VV + VH, dB) acquired the
same morning as the Sentinel-2 image, resampled to the Sentinel-2 grid.

1. How a flood looks in SAR, and what VV and VH show.
2. Histograms: full scene vs a small area → bimodality.
3. Pick a threshold by eye → overlap between water and land.
4. Otsu, why it fails on the full scene, and two fixes: **Edge Otsu** and **BMax Otsu**.
5. Removing false positives with terrain: **HAND**.

**Google Colab:** run the next cell first (~1 min). It downloads the course code, data and models and installs
ml4floods; if Colab disconnects, just run it again. On your own computer it does nothing.

In [ ]:
import sys
if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    COURSE = Path("/content/geomundus-ml4floods")
    if not COURSE.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/kipoju/geomundus-ml4floods", str(COURSE)], check=True)
    sys.path.insert(0, str(COURSE / "notebooks"))
    import colab_setup
    colab_setup.setup()

In [ ]:
import sys
sys.path.append(".")

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from georeader import plot, read
from georeader.save import save_cog
from matplotlib.patches import Rectangle
from skimage.exposure import rescale_intensity
from skimage.filters import threshold_otsu
from course_utils import (BOX_BOUNDS, BOX_CRS, EVENT, ROOT, binary_metrics, bmax_otsu, composite, confusion_legend,
                          confusion_overlay, edge_otsu, find_event, hist_by_class, interact, read_event, save_fig)

data_dir, split = find_event(EVENT)
d = read_event(data_dir, split, EVENT)
gt = d["gt"]
vv, vh = d["s1"]          # GeoTensors (y, x) in dB, on the Sentinel-2 grid
print("S2:", d["meta"]["s2_date"])
for sc in d["s1meta"]["scenes"]:
    print("S1:", sc["utcdatetime"], sc["orbit_pass"], f"({sc['hours_to_s2']:+.1f} h from S2)")

# Reference: water / land from Copernicus EMS. We evaluate on valid pixels outside the S2 clouds
# (same pixels as for the optical indices, so numbers are comparable).
ref = gt[1] == 2                      # gt band 1: {0: invalid, 1: land, 2: water}
valid = (gt[1] != 0) & (gt[0] != 2)   # gt band 0: {0: invalid, 1: clear, 2: cloud}
VV_KW = dict(cmap="gray", vmin=-25, vmax=5)
VH_KW = dict(cmap="gray", vmin=-30, vmax=-5)
XLABEL = r"VV backscatter $\sigma^0$ [dB]"

## 1. A flood seen by SAR
Calm water is a smooth surface: the radar pulse is reflected away from the sensor (specular
reflection) → **water is dark**. Compare with the optical image: SAR also sees below the clouds.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 6))
axs[0].imshow(composite(d["s2"], ("B11", "B8", "B4"), 4500)); axs[0].set_title("Sentinel-2 (B11, B8, B4)")
axs[1].imshow(vv, **VV_KW);                                   axs[1].set_title("Sentinel-1 VV [dB]")
axs[2].imshow(vh, **VH_KW);                                   axs[2].set_title("Sentinel-1 VH [dB]")
for ax in axs:
    ax.set_axis_off()
plt.tight_layout()

### VV and VH on another event of the same flood
EMSR271_04STEFANOVIKIO (training event, Sentinel-1 of 1 March 2018): sea, coast, fields, towns and
flooded fields. **VV** (co-polarised) is more sensitive to surface roughness; **VH** (cross-polarised)
to volume scattering (vegetation). The colour composite (R, G, B) = (VV, VH, VV − VH) combines both.

In [ ]:
STEF = "EMSR271_04STEFANOVIKIO_DEL_v1"
try:
    ds = read_event(*find_event(STEF), STEF)
    svv, svh = ds["s1"].values[:, :, 900:]      # eastern part: coast + flood (plain numpy: only for display)
    # dB ranges -> [0, 1] (values outside the range are clipped)
    sar_rgb = np.stack([rescale_intensity(svv, in_range=(-25, 0), out_range=(0, 1)),
                        rescale_intensity(svh, in_range=(-30, -5), out_range=(0, 1)),
                        rescale_intensity(svv - svh, in_range=(0, 15), out_range=(0, 1))], axis=-1)
    fig, axs = plt.subplots(1, 4, figsize=(14, 5.2))
    axs[0].imshow(composite(ds["s2"][:, :, 900:], ("B11", "B8", "B4"), 4500))
    axs[0].set_title("Sentinel-2 (B11, B8, B4)", fontsize=12)
    axs[1].imshow(svv, **VV_KW);  axs[1].set_title("VV [dB]", fontsize=12)
    axs[2].imshow(svh, **VH_KW);  axs[2].set_title("VH [dB]", fontsize=12)
    axs[3].imshow(sar_rgb);       axs[3].set_title("(R, G, B) = (VV, VH, VV − VH)", fontsize=12)
    for ax in axs:
        ax.set_axis_off()
    plt.tight_layout()
    save_fig(fig, "sar_polarisation")
except FileNotFoundError:
    print(f"{STEF} not in the data package: skipping")

## 2. Histograms: full scene vs a small area
**Predict first:** if water is dark and land is bright, what shape should the VV histogram have?

In [ ]:
# red box: crop the GeoTensors by UTM bounds with georeader (same box for the image and the reference)
vv_box = read.read_from_bounds(vv, BOX_BOUNDS, crs_bounds=BOX_CRS)
gt_box = read.read_from_bounds(gt, BOX_BOUNDS, crs_bounds=BOX_CRS)
ref_box, valid_box = gt_box[1] == 2, (gt_box[1] != 0) & (gt_box[0] != 2)
win = read.window_from_bounds(d["s1"], BOX_BOUNDS, crs_bounds=BOX_CRS)   # position of the box in pixels
BINS = np.linspace(-30, 5, 141)

fig = plt.figure(figsize=(14, 5.9))
gs = fig.add_gridspec(2, 2, width_ratios=[1.45, 1], hspace=0.45, wspace=0.08)
ax = fig.add_subplot(gs[:, 0])
ax.imshow(vv, **VV_KW)
ax.add_patch(Rectangle((win.col_off, win.row_off), win.width, win.height, lw=2.5, edgecolor="r", facecolor="none"))
ax.set_title("Sentinel-1 VV — FARKADONA", fontsize=13)
ax.set_axis_off()
for i, (name, values, col) in enumerate([("Full scene", vv, "k"), ("Red box", vv_box, "r")]):
    ax = fig.add_subplot(gs[i, 1])
    ax.hist(np.asarray(values).ravel(), bins=BINS, color="midnightblue")
    ax.set_title(name, fontsize=12, color=col)
    ax.set_xlabel(XLABEL, fontsize=10)
    ax.set_yticks([])
save_fig(fig, "sar_flood_hist")

In the full scene water is only a small fraction of the pixels → its mode is hidden in the tail of the
land mode. In the red box water and land are balanced → **bimodal** histogram: we can put a threshold
in the valley between both modes.

## 3. A threshold by eye — and the overlap between classes
Now we colour the histogram of the box with the reference: **blue = water**, **orange = land**,
and map the errors of the threshold: **navy** = water hit, **red** = false alarm, **green** = missed water.

In [ ]:
T_EYE = -17.0   # <- your threshold from the valley of the red-box histogram


def show_sar_threshold(t=-17.0, area="red box", figsize=(17, 5.5)):
    img, r, v = (vv_box, ref_box, valid_box) if area == "red box" else (vv, ref, valid)
    pred = img < t
    m = binary_metrics(pred, r, v)
    fig, axs = plt.subplots(1, 2, figsize=figsize, gridspec_kw={"width_ratios": [1, 1.1]})
    hist_by_class(img, r & v, ~r & v, t=t, ax=axs[0], bins=BINS, xlabel=XLABEL)
    axs[0].set_title(f"Histogram by reference class ({area})")
    axs[1].imshow(confusion_overlay(pred, r, v), interpolation="nearest")
    axs[1].set_title(f"VV < {t:.1f} dB — P {m['precision']:.2f} · R {m['recall']:.2f} · IoU {m['IoU']:.2f}")
    axs[1].set_axis_off()
    confusion_legend(axs[1], loc="lower right")
    plt.tight_layout()
    return fig


fig = show_sar_threshold(T_EYE, figsize=(14, 5.2))
save_fig(fig, "sar_threshold")

The two classes **overlap**: land pixels darker than the threshold become false alarms (smooth fields,
radar shadows), water pixels brighter than it are missed (wind-roughened water, flooded vegetation).
No single threshold can be perfect.

### Try it yourself
Move the threshold and look at the histogram and at the map of errors.

In [ ]:
def try_threshold(t=-17.0, area="red box"):
    show_sar_threshold(t, area)
    plt.show()


interact(try_threshold,
         t=widgets.FloatSlider(value=-17.0, min=-25, max=-5, step=0.5, description="threshold"),
         area=widgets.ToggleButtons(options=["red box", "full scene"]))

## 4. Otsu: finding the threshold automatically
**Otsu (1979)** tries every threshold t and keeps the one that maximises the **between-class variance**
σ²_B(t) = p_water · p_land · (μ_water − μ_land)², i.e. two groups as separated (and compact) as possible.
It works when the histogram is bimodal.
Below we plot σ²_B(t) to see it; in practice we just call `skimage.filters.threshold_otsu`.

We wrap it in a function for **numpy arrays**: image in → water mask out. Because a GeoTensor is a numpy array,
the same function works on our georeferenced Sentinel-1 image and the mask comes back as a GeoTensor, on the
same grid, ready to be saved or vectorised (golden rule of notebook 01a).

In [ ]:
def otsu_numpy(x: np.ndarray) -> np.ndarray:
    """Water mask of a SAR image in dB: pixels darker than the Otsu threshold of the image."""
    t = threshold_otsu(np.asarray(x)[np.isfinite(x)])   # threshold from the valid pixels
    return x < t


flood_mask_box = otsu_numpy(vv_box)
print(type(vv_box).__name__, "→", type(flood_mask_box).__name__, flood_mask_box.dtype,
      "| same grid:", flood_mask_box.same_extent(vv_box))

fig, axs = plt.subplots(1, 2, figsize=(14, 4.6))
plot.show(vv_box, ax=axs[0], add_scalebar=True, **VV_KW); axs[0].set_title("VV, red box [dB]")
plot.show(flood_mask_box, ax=axs[1], add_scalebar=True, cmap="Blues"); axs[1].set_title("otsu_numpy(VV): water")
plt.tight_layout()

In [ ]:
def between_class_variance(values, thresholds):
    values = np.asarray(values).ravel()
    out = []
    for t in thresholds:
        w = values < t
        p = w.mean()
        out.append(0 if p in (0, 1) else p * (1 - p) * (values[w].mean() - values[~w].mean()) ** 2)
    return np.array(out)


ts = np.linspace(-28, 0, 141)
t_box = threshold_otsu(vv_box.values)
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.hist(np.asarray(vv_box).ravel(), bins=BINS, color="midnightblue", alpha=0.8)
ax.set_xlabel(XLABEL, fontsize=12)
ax.set_ylabel("pixels", fontsize=12)
ax2 = ax.twinx()
ax2.plot(ts, between_class_variance(vv_box, ts), color="tab:red", lw=2.5)
ax2.set_ylabel(r"between-class variance $\sigma^2_B(t)$", color="tab:red", fontsize=12)
ax2.set_yticks([])
ax.axvline(t_box, color="tab:red", ls="--", lw=1.5)
ax.annotate(f"Otsu = {t_box:.1f} dB", (t_box, ax.get_ylim()[1] * 0.45), xytext=(-8, 0), ha="right",
            textcoords="offset points", color="tab:red", fontsize=13, fontweight="bold")
ax.set_title("Red box", fontsize=13)
plt.tight_layout()
save_fig(fig, "sar_otsu_curve")

**The same function on the full scene.** ❓ Predict first: will the threshold be similar?

In [ ]:
flood_mask_full = otsu_numpy(vv)
t_full = threshold_otsu(vv.values)
fig, axs = plt.subplots(1, 2, figsize=(14, 5.2))
for ax, (name, img, r, v, t) in zip(axs, [("Red box", vv_box, ref_box, valid_box, t_box),
                                          ("Full scene", vv, ref, valid, t_full)]):
    hist_by_class(img, r & v, ~r & v, t=t, ax=ax, bins=BINS, xlabel=XLABEL)
    ax.set_title(f"{name}: Otsu = {t:.1f} dB", fontsize=13)
plt.tight_layout()
save_fig(fig, "sar_otsu_box_vs_full")
print(f"Otsu red box: {t_box:.1f} dB | Otsu full scene: {t_full:.1f} dB")

On the full scene Otsu splits the *land* mode in two (the histogram is not bimodal) → a much too high
threshold and lots of false alarms. We cannot draw a red box by hand for every image… Two automatic
fixes (Markert et al., 2020, https://doi.org/10.3390/rs12152469):

- **Edge Otsu:** make a rough water mask with an initial threshold, detect its **edges** (Canny) and
  compute Otsu only on pixels **near the edges** → there, water and land are balanced.
- **BMax Otsu** (Cao et al., 2019): split the image in **tiles**, measure how bimodal each tile is
  (BMax = normalised between-class variance) and compute Otsu only on the **bimodal tiles**.

In [ ]:
T0 = -16.0   # initial rough threshold, only used to find edges / test bimodality
t_edge, edge_buffer = edge_otsu(vv, t0=T0)

fig, axs = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})
axs[0].imshow(vv, **VV_KW)
axs[0].imshow(np.ma.masked_where(~edge_buffer, edge_buffer), cmap="autumn", alpha=0.5, interpolation="nearest")
axs[0].set_title(f"Buffered edges ({100 * edge_buffer.mean():.0f} % of the pixels)", fontsize=12)
axs[0].set_axis_off()
hist_by_class(vv, ref & valid & edge_buffer, ~ref & valid & edge_buffer, t=t_edge, ax=axs[1], bins=BINS,
              xlabel=XLABEL)
axs[1].set_title(f"Histogram near the edges: Otsu = {t_edge:.1f} dB", fontsize=12)
plt.tight_layout()
save_fig(fig, "sar_edge_otsu")

In [ ]:
TILE = 128   # pixels (1.28 km)
t_bmax, bm, selected = bmax_otsu(vv, t0=T0, tile=TILE, bmax_threshold=0.75)

fig, axs = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})
axs[0].imshow(vv, **VV_KW)
im = axs[0].imshow(bm, cmap="viridis", vmin=0, vmax=1, alpha=0.45, interpolation="nearest",
                   extent=(0, bm.shape[1] * TILE, bm.shape[0] * TILE, 0))   # one BMax value per tile
axs[0].contour(selected, levels=[0.5], colors="r", linewidths=1.5)
axs[0].set_title(f"BMax per tile: {(bm > 0.75).sum()} of {bm.size} bimodal (red)", fontsize=12)
axs[0].set_axis_off()
fig.colorbar(im, ax=axs[0], shrink=0.8, label="BMax", pad=0.02)
hist_by_class(vv, ref & valid & selected, ~ref & valid & selected, t=t_bmax, ax=axs[1], bins=BINS, xlabel=XLABEL)
axs[1].set_title(f"Histogram of the bimodal tiles: Otsu = {t_bmax:.1f} dB", fontsize=12)
plt.tight_layout()
save_fig(fig, "sar_bmax_otsu")

### Comparison on the full scene

In [ ]:
methods = {"By eye": T_EYE, "Otsu (full scene)": t_full, "Otsu (red box)": t_box,
           "Edge Otsu": t_edge, "BMax Otsu": t_bmax}
rows = {name: {"threshold [dB]": t, **binary_metrics(vv < t, ref, valid)} for name, t in methods.items()}
assert np.array_equal(vv < t_full, flood_mask_full)   # "Otsu (full scene)" = otsu_numpy(vv)
table = pd.DataFrame(rows).T.drop(columns="OA").astype(float).round(2)
table

In [ ]:
show = ["Otsu (full scene)", "Edge Otsu", "BMax Otsu"]
fig, axs = plt.subplots(1, len(show), figsize=(14, 4.6))
for ax, name in zip(axs, show):
    ax.imshow(confusion_overlay(vv < methods[name], ref, valid), interpolation="nearest")
    ax.set_title(f"{name} ({methods[name]:.1f} dB)\nP {table.loc[name, 'precision']:.2f} · "
                 f"R {table.loc[name, 'recall']:.2f} · IoU {table.loc[name, 'IoU']:.2f}", fontsize=12)
    ax.set_axis_off()
confusion_legend(axs[0], fontsize=8)
plt.tight_layout()
save_fig(fig, "sar_methods_compare")

## 5. Using terrain: HAND
**HAND** (Height Above Nearest Drainage) = how many metres a pixel is above the closest river.
A pixel 50 m above the river is very unlikely to be flooded → we remove the "water" detected there
(mostly **radar shadows** on the slopes of the hills).

In [ ]:
hand = d["hand"]   # GeoTensor on the same grid (MERIT Hydro, resampled from ~90 m to 10 m)
HAND_MAX = 15   # metres
t_best = t_edge
pred = vv < t_best                    # GeoTensor
pred_hand = pred & (hand < HAND_MAX)  # GeoTensor: logic between rasters on the same grid

fig, axs = plt.subplots(1, 3, figsize=(14, 4.4))
im = axs[0].imshow(hand, cmap="terrain", vmin=0, vmax=100)
fig.colorbar(im, ax=axs[0], shrink=0.85, label="HAND [m]", pad=0.02)
axs[0].set_title("HAND (MERIT Hydro)", fontsize=12)
for ax, (name, p) in zip(axs[1:], [(f"Edge Otsu (VV < {t_best:.1f} dB)", pred),
                                   (f"+ HAND < {HAND_MAX} m", pred_hand)]):
    m = binary_metrics(p, ref, valid)
    ax.imshow(confusion_overlay(p, ref, valid), interpolation="nearest")
    ax.set_title(f"{name}\nP {m['precision']:.2f} · R {m['recall']:.2f} · IoU {m['IoU']:.2f}", fontsize=12)
for ax in axs:
    ax.set_axis_off()
confusion_legend(axs[2], fontsize=7)
plt.tight_layout()
save_fig(fig, "sar_hand")

pd.DataFrame({name: binary_metrics(p, ref, valid) for name, p in
              [("Edge Otsu", pred), (f"Edge Otsu + HAND<{HAND_MAX} m", pred_hand)]}).T.round(2)

### Our SAR flood map is a georeferenced raster
Every step kept the georeference, so the result can go straight into a GIS:

In [ ]:
out_dir = ROOT / "outputs"
out_dir.mkdir(exist_ok=True)
out_file = out_dir / f"{EVENT}_S1_edgeotsu_hand.tif"
save_cog(pred_hand.astype(np.uint8), str(out_file), descriptions=["water"])
print("saved", out_file, "|", pred_hand.crs, pred_hand.bounds)

**Take-aways**
- SAR sees water through clouds, day and night; calm water is dark.
- A single threshold works only where the histogram is bimodal → Otsu on the full scene fails;
  Edge Otsu and BMax Otsu look for the bimodal parts automatically.
- Class overlap is intrinsic: smooth fields and shadows look like water. HAND removes the ones on
  slopes, but not those on the flat floodplain → we need **context**, not just one pixel value:
  this is what deep learning models learn from data.